# Notebook 5 (Fixed): Model Training and Bundle Export

This fixed notebook repairs the original Notebook 5 workflow in depth:
- Removes duplicated/fragile blocks and replaces them with one deterministic pipeline.
- Handles AMR gene-column variants (`Gene symbol` or `NAME`).
- Handles missing files gracefully and auto-loads from `amr_results/` (or `amr_results.zip` if needed).
- Trains multiple models per antibiotic, selects best by AUC, and saves:
  - `*_df.csv`
  - `*_model_bundle.pkl`
  - `model_training_summary.csv`


In [5]:
from pathlib import Path
import zipfile
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import train_test_split

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TARGET_ANTIBIOTICS = ['gentamicin', 'ciprofloxacin', 'meropenem']

print('xgboost available:', XGBOOST_AVAILABLE)


xgboost available: True


In [6]:
def _parse_genome_id_from_filename(p: Path):
    stem = p.stem.replace('_amr', '')
    try:
        return float(stem)
    except ValueError:
        return None


def _discover_amr_tsv_files(amr_dir=Path('../amr_results'), zip_path=Path('../amr_results.zip')):
    amr_dir = Path(amr_dir)
    if not amr_dir.exists() and zip_path.exists():
        amr_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(amr_dir)

    tsv_files = sorted(amr_dir.glob('*.tsv'))
    if not tsv_files:
        # Handle nested extractions like amr_results/amr_results/*.tsv
        nested = amr_dir / 'amr_results'
        if nested.exists():
            tsv_files = sorted(nested.glob('*.tsv'))

    return tsv_files


def _extract_genes_from_tsv(tsv_path: Path):
    df = pd.read_csv(tsv_path, sep='\t')
    for col in ['Gene symbol', 'NAME', 'Gene', 'gene']:
        if col in df.columns:
            vals = df[col].dropna().astype(str).str.strip()
            vals = vals[vals != '']
            return set(vals.unique())
    return set()


def build_gene_matrix(amr_tsv_files):
    all_genes = set()
    genome_genes_map = {}

    for tsv in amr_tsv_files:
        gid = _parse_genome_id_from_filename(tsv)
        if gid is None:
            continue
        try:
            genes = _extract_genes_from_tsv(tsv)
        except pd.errors.EmptyDataError:
            genes = set()
        except Exception:
            genes = set()

        genome_genes_map[gid] = genes
        all_genes.update(genes)

    genes_list = sorted(all_genes)
    rows = []
    for gid, genes in genome_genes_map.items():
        rows.append([gid] + [1 if g in genes else 0 for g in genes_list])

    if not rows:
        return pd.DataFrame(columns=['Genome ID'])

    df_genes = pd.DataFrame(rows, columns=['Genome ID'] + [f'gene_{g}' for g in genes_list])
    df_genes['Genome ID'] = pd.to_numeric(df_genes['Genome ID'], errors='coerce')
    df_genes = df_genes.dropna(subset=['Genome ID']).drop_duplicates(subset=['Genome ID'])
    return df_genes


def build_model_candidates():
    candidates = {
        'logistic_regression': LogisticRegression(
            random_state=RANDOM_STATE, solver='liblinear', penalty='l1', C=1.0, max_iter=1000
        ),
        'random_forest': RandomForestClassifier(
            n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
        ),
    }

    if XGBOOST_AVAILABLE:
        candidates['xgboost'] = XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    return candidates


In [7]:
refined_path = Path('../data/processed/refined_data.csv')
if not refined_path.exists():
    raise FileNotFoundError('refined_data.csv not found. Run Notebook 2 fixed first.')

df_phenotype = pd.read_csv(refined_path)
if 'Genome ID' not in df_phenotype.columns:
    raise ValueError('refined_data.csv is missing required column: Genome ID')

df_phenotype['Genome ID'] = pd.to_numeric(df_phenotype['Genome ID'], errors='coerce')
df_phenotype = df_phenotype.dropna(subset=['Genome ID']).copy()

amr_tsv_files = _discover_amr_tsv_files()
if not amr_tsv_files:
    raise FileNotFoundError('No AMR TSV files found in ../amr_results/ (or ../amr_results.zip).')

print('AMR TSV files found:', len(amr_tsv_files))
df_genes = build_gene_matrix(amr_tsv_files)
print('df_genes shape:', df_genes.shape)

if df_genes.shape[1] <= 1:
    raise ValueError('No AMR gene features were extracted from TSV files.')

df_merged = df_phenotype.merge(df_genes, on='Genome ID', how='inner')
print('Merged shape:', df_merged.shape)

if df_merged.empty:
    raise ValueError('Merged phenotype+gene dataset is empty. Check Genome ID overlap.')

summary_rows = []

for antibiotic in TARGET_ANTIBIOTICS:
    print(f'\n=== {antibiotic} ===')
    df_a = df_merged[df_merged['Antibiotic'].astype(str).str.lower() == antibiotic].copy()

    if df_a.empty:
        print('No rows found for this antibiotic. Skipping.')
        continue

    df_a = df_a[df_a['Resistant Phenotype'].isin(['Resistant', 'Susceptible'])].copy()
    if df_a.empty:
        print('No Resistant/Susceptible rows after filtering. Skipping.')
        continue

    df_a['Resistant_Binary'] = (df_a['Resistant Phenotype'] == 'Resistant').astype(int)

    gene_cols = [c for c in df_a.columns if c.startswith('gene_')]
    X = df_a[gene_cols].copy()
    y = df_a['Resistant_Binary'].copy()

    if X.empty or y.nunique() < 2:
        print('Insufficient features or only one class. Skipping.')
        continue

    class_counts = y.value_counts()
    use_stratify = class_counts.min() >= 2
    stratify_arg = y if use_stratify else None

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=stratify_arg
    )

    candidates = build_model_candidates()
    best_name = None
    best_model = None
    best_auc = -1.0
    best_mcc = np.nan
    per_model_scores = {}

    for name, model in candidates.items():
        model.fit(X_train, y_train)
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)

        auc_val = roc_auc_score(y_test, y_proba)
        mcc_val = matthews_corrcoef(y_test, y_pred)
        per_model_scores[name] = {'auc': float(auc_val), 'mcc': float(mcc_val)}

        if auc_val > best_auc:
            best_auc = float(auc_val)
            best_mcc = float(mcc_val)
            best_name = name
            best_model = model

    if best_model is None:
        print('No model trained successfully. Skipping.')
        continue

    out_df = df_a[['Genome ID', 'Antibiotic', 'Resistant Phenotype'] + gene_cols + ['Resistant_Binary']].copy()
    df_path = Path(f'../data/features/{antibiotic}_df.csv')
    out_df.to_csv(df_path, index=False)

    bundle = {
        'model': best_model,
        'model_name': best_name,
        'feature_cols': gene_cols,
        'antibiotic': antibiotic,
        'scores': per_model_scores,
        'best_auc': best_auc,
        'best_mcc': best_mcc,
        'train_size': int(len(X_train)),
        'test_size': int(len(X_test)),
        'random_state': RANDOM_STATE,
    }

    bundle_path = Path(f'../models/{antibiotic}_model_bundle.pkl')
    joblib.dump(bundle, bundle_path)

    summary_rows.append({
        'antibiotic': antibiotic,
        'rows': int(len(df_a)),
        'n_features': int(len(gene_cols)),
        'best_model': best_name,
        'best_auc': best_auc,
        'best_mcc': best_mcc,
        'df_file': str(df_path),
        'bundle_file': str(bundle_path),
    })

    print(f'Saved: {df_path} and {bundle_path}')
    print(f'Best model: {best_name} | AUC={best_auc:.4f} | MCC={best_mcc:.4f}')

summary_df = pd.DataFrame(summary_rows)
if summary_df.empty:
    raise RuntimeError('No antibiotic model was trained. Check class balance and available rows.')

summary_df.to_csv('../data/processed/model_training_summary.csv', index=False)
summary_df


AMR TSV files found: 274
df_genes shape: (274, 312)
Merged shape: (3604, 327)

=== gentamicin ===
Saved: ../data/features/gentamicin_df.csv and ../models/gentamicin_model_bundle.pkl
Best model: logistic_regression | AUC=1.0000 | MCC=0.9129

=== ciprofloxacin ===
Saved: ../data/features/ciprofloxacin_df.csv and ../models/ciprofloxacin_model_bundle.pkl
Best model: logistic_regression | AUC=0.9750 | MCC=0.8919

=== meropenem ===
Saved: ../data/features/meropenem_df.csv and ../models/meropenem_model_bundle.pkl
Best model: logistic_regression | AUC=0.9658 | MCC=0.8050


,antibiotic,rows,n_features,best_model,best_auc,best_mcc,df_file,bundle_file
0,gentamicin,277,311,logistic_regression,1.000000,0.912871,../data/features/gentamicin_df.csv,../models/gentamicin_model_bundle.pkl
1,ciprofloxacin,266,311,logistic_regression,0.975000,0.891883,../data/features/ciprofloxacin_df.csv,../models/ciprofloxacin_model_bundle.pkl
2,meropenem,239,311,logistic_regression,0.965812,0.805011,../data/features/meropenem_df.csv,../models/meropenem_model_bundle.pkl


In [8]:
for antibiotic in TARGET_ANTIBIOTICS:
    df_file = Path(f'../data/features/{antibiotic}_df.csv')
    bundle_file = Path(f'../models/{antibiotic}_model_bundle.pkl')
    print(antibiotic, 'df:', df_file.exists(), '| bundle:', bundle_file.exists())

print('summary exists:', Path('../data/processed/model_training_summary.csv').exists())


gentamicin df: True | bundle: True
ciprofloxacin df: True | bundle: True
meropenem df: True | bundle: True
summary exists: True
